# GBPUSD V. EURUSD — TUNING

Short, focused tune on **3 months** of data (Jan–Mar 2024). The goal is to find an engine configuration where MS-AR provably beats AR (the hard-kill baseline that uses the same Markov model).

All grid cells use the new `objective='ms_minus_ar'` from `wfo.py`, which maximises `Sharpe(MS_AR) − Sharpe(AR)` directly so the comparison is fair.

Outputs:
- per-cell metrics table
- heatmap of `MS−AR Sharpe edge` over the grid
- best config saved to `code/tune/audnzd_best.json` for use in the main notebook

### GOOGLE DRIVE

In [ ]:
# Colab Setup Script
from google.colab import drive
import sys, os, shutil, subprocess

drive.mount('/content/drive')

REPO_ROOT  = '/content/stk-mat2011'
REPO_DATA  = f'{REPO_ROOT}/code/data/processed'
DRIVE_DATA = '/content/drive/MyDrive/GITHUB-COPILOT/stk-mat2011/data/processed'

# CLONE (or pull if already cloned this session)
if not os.path.isdir(REPO_ROOT):
    subprocess.run(['git', 'clone', 'https://github.com/egil10/stk-mat2011.git', REPO_ROOT], check=True)
else:
    subprocess.run(['git', '-C', REPO_ROOT, 'pull'], check=True)

# REPLACE empty data dir from clone with symlink to Drive
if os.path.isdir(REPO_DATA) and not os.path.islink(REPO_DATA):
    shutil.rmtree(REPO_DATA)
if os.path.islink(REPO_DATA) and not os.path.exists(REPO_DATA):
    os.unlink(REPO_DATA)
if not os.path.islink(REPO_DATA):
    os.symlink(DRIVE_DATA, REPO_DATA)

# IMPORTS + working directory to match local notebook environment
sys.path.append(f'{REPO_ROOT}/code/scripts')
os.chdir(f'{REPO_ROOT}/code/tune')

# SANITY
print(f"CWD:           {os.getcwd()}")
print(f"wfo.py:        {os.path.isfile(f'{REPO_ROOT}/code/scripts/wfo.py')}")
print(f"Data symlink:  {os.path.islink(REPO_DATA)} -> {os.readlink(REPO_DATA) if os.path.islink(REPO_DATA) else 'N/A'}")
print(f"Parquet count: {len([f for f in os.listdir(REPO_DATA) if f.endswith('.parquet')])}")
print(f"Path test:     {os.path.exists('../data/processed/gbpusd_dukascopy_ask_202401.parquet')}")

### IMPORT

In [ ]:
%%capture
!pip install arch optuna

import os, sys, json, warnings, itertools
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
warnings.filterwarnings("ignore", module="statsmodels.tsa.base.tsa_model")

sys.path.append(os.path.abspath('../scripts'))

from spread import SPREAD
from screener import SCREENER
from engine import ENGINE
from backtester import BACKTESTER
from descriptive import DESCRIPTIVE
from wfo import WFO

### PARAMS — short tuning window

The point of this notebook is fast iteration. We:
- restrict the backtest to **3 months** (Jan–Mar 2024)
- use **2 months train / 1 month test** WFO so each grid cell still produces real OOS metrics
- keep `n_trials` modest (~50) per WFO window — tuning quality dominates over Optuna depth here

In [ ]:
# ==========================================
# 1. PAIR + DATA WINDOW
# ==========================================
NAME_A, NAME_B = "GBPUSD", "EURUSD"
PAIR_NAME = f"{NAME_A}_{NAME_B}"

# 3-month tuning window
TUNE_MONTHS = ["202401", "202402", "202403"]

# Session filter (full 24h to maximise sample size for the HMM)
START_HOUR, END_HOUR = 0, 24
THRESHOLD = 1000
AGG_TYPE  = 'tick'

# ==========================================
# 2. ENGINE BASE PARAMS (everything outside the grid)
# ==========================================
TRAIN_DAYS = 20            # daily rolling history per fold (smaller = faster, still stable)
RANDOM_SEED = 42

# ==========================================
# 3. WFO BASE PARAMS
# ==========================================
VAL_MONTHS  = 2            # 2 months train -> 1 month test fits inside our 3-month window
TEST_MONTHS = 1
N_TRIALS    = 50           # per WFO window
OBJECTIVE   = 'ms_minus_ar'

# ==========================================
# 4. THE GRID
# ==========================================
# Compact 2 x 2 x 2 x 2 = 16 cells. Run sequentially; each cell is ~1-3 min on Colab.
GRID = {
    'k_regimes':     [2, 3],
    'winsorize_std': [3.0, 5.0],
    'coint_window':  [200, 400],
    'z_window':      [80, 150],
}

# Output paths
TUNE_OUTDIR  = os.path.join(os.path.dirname(os.path.dirname(DRIVE_DATA)), "tune", PAIR_NAME)
os.makedirs(TUNE_OUTDIR, exist_ok=True)
RESULTS_CSV  = os.path.join(TUNE_OUTDIR, f"{PAIR_NAME}_grid_results.csv")
BEST_JSON    = os.path.join(TUNE_OUTDIR, f"{PAIR_NAME}_best.json")
print(f"Tune outputs -> {TUNE_OUTDIR}")
print(f"Grid size: {np.prod([len(v) for v in GRID.values()])} cells")

### LOAD 3-MONTH SLICE

In [ ]:
DATA_DIR = "../data/processed"

def make_files(name_a, name_b, months):
    a, b = name_a.lower(), name_b.lower()
    return [
        [f"{DATA_DIR}/{a}_dukascopy_ask_{m}.parquet" for m in months],
        [f"{DATA_DIR}/{a}_dukascopy_bid_{m}.parquet" for m in months],
        [f"{DATA_DIR}/{b}_dukascopy_ask_{m}.parquet" for m in months],
        [f"{DATA_DIR}/{b}_dukascopy_bid_{m}.parquet" for m in months],
    ]

builder = SPREAD(
    agg_type=AGG_TYPE,
    threshold=THRESHOLD,
    active_hours=(START_HOUR, END_HOUR),
)
df = builder.build(make_files(NAME_A, NAME_B, TUNE_MONTHS))

print(f"\nFrame  : {len(df):,} bars")
print(f"Period : {df.index.min()}  ->  {df.index.max()}")
print(f"Days   : {df.index.normalize().unique().shape[0]}")

### QUICK SCREENER

A single-shot cointegration sanity check on the 3-month slice. If the pair is wildly non-cointegrated *over this window*, the rest of the tune is moot — we'd want a different window or a different pair.

In [ ]:
screener = SCREENER(df['Asset_A'], df['Asset_B'])
p_val, hl = screener.generate_report(rolling_window=2000, rolling_step=200)
print(f"\nFull-window p-value: {p_val:.4f}    half-life: {hl:.1f} bars")

### GRID SWEEP

For each cell `(k_regimes, winsorize_std, coint_window, z_window)`:

1. Run `ENGINE.walk_forward` on the 3-month slice with that engine config.
2. Run `WFO.run_wfo` with `objective='ms_minus_ar'`.
3. Compute `Sharpe_MS`, `Sharpe_AR`, `Sharpe_diff`, total PnL bps for each strategy, and a regime-stability metric (count of MR/DR label-flips between consecutive folds).

The cell that maximises `Sharpe_diff` while keeping `Sharpe_MS` non-negative becomes the recommended config.

In [ ]:
ANN_FACTOR = 252 * 24 * 60   # bars-per-year for high-freq Sharpe annualisation

def _sharpe(r):
    r = r.fillna(0)
    s = r.std()
    return float(r.mean() / s * np.sqrt(ANN_FACTOR)) if s and np.isfinite(s) else 0.0

def evaluate_cell(df, k_regimes, winsorize_std, coint_window, z_window,
                  train_days=TRAIN_DAYS, val_months=VAL_MONTHS,
                  test_months=TEST_MONTHS, n_trials=N_TRIALS,
                  objective=OBJECTIVE, verbose=False):
    """One grid cell: ENGINE.walk_forward -> WFO -> per-strategy metrics."""

    live, params = ENGINE.walk_forward(
        df=df,
        train_days=train_days,
        coint_window=coint_window,
        z_window=z_window,
        k_regimes=k_regimes,
        winsorize_std=winsorize_std,
        scaling=10000,
        print_freq=10**6,        # silence per-day prints
    )
    if len(live) == 0:
        return None

    wfo_obj = WFO(live, flatten_eod=False)
    oos = wfo_obj.run_wfo(
        val_months=val_months,
        test_months=test_months,
        n_trials=n_trials,
        objective=objective,
        verbose=verbose,
    )

    out = {
        'k_regimes': k_regimes, 'winsorize_std': winsorize_std,
        'coint_window': coint_window, 'z_window': z_window,
        'oos_rows': len(oos), 'fold_count': len(params),
    }
    for s in ('Baseline', 'AR', 'MS_AR'):
        r = oos[f'Return_{s}'].fillna(0)
        out[f'{s}_sharpe']    = _sharpe(r)
        out[f'{s}_pnl_bps']   = float(r.sum() * 1e4)
        out[f'{s}_n_trades']  = int((oos[f'Target_{s}'].diff().abs() > 0).sum() / 2)
        out[f'{s}_exposure']  = float((r != 0).mean())

    out['Sharpe_diff_MS_AR']  = out['MS_AR_sharpe'] - out['AR_sharpe']
    out['PnL_diff_MS_AR_bps'] = out['MS_AR_pnl_bps'] - out['AR_pnl_bps']

    # Regime stability: how often does the MR-regime sigma flip relative to DR
    # across consecutive WFO folds. High flip-count -> noisy/identifiable model.
    if {'Safe_Variance', 'Danger_Variance'}.issubset(params.columns):
        ratio = (params['Safe_Variance'] / params['Danger_Variance']).dropna()
        out['regime_var_ratio_med'] = float(ratio.median())
    else:
        out['regime_var_ratio_med'] = np.nan

    return out

In [ ]:
cells = list(itertools.product(
    GRID['k_regimes'], GRID['winsorize_std'],
    GRID['coint_window'], GRID['z_window'],
))
print(f"Running grid: {len(cells)} cells\n")

rows = []
for i, (k, w, c, z) in enumerate(cells, 1):
    print(f"[{i:2d}/{len(cells)}] k={k}  winsor={w}  coint={c}  z={z}", flush=True)
    try:
        out = evaluate_cell(df, k_regimes=k, winsorize_std=w,
                            coint_window=c, z_window=z, verbose=False)
    except Exception as e:
        print(f"   FAILED: {type(e).__name__}: {e}")
        out = None

    if out is None:
        continue

    print(f"   MS  Sharpe={out['MS_AR_sharpe']:+.2f}  PnL={out['MS_AR_pnl_bps']:+8.1f}bps  trades={out['MS_AR_n_trades']:>3}")
    print(f"   AR  Sharpe={out['AR_sharpe']:+.2f}  PnL={out['AR_pnl_bps']:+8.1f}bps  trades={out['AR_n_trades']:>3}")
    print(f"   EDGE  Sharpe_diff={out['Sharpe_diff_MS_AR']:+.2f}  PnL_diff={out['PnL_diff_MS_AR_bps']:+.1f}bps\n")
    rows.append(out)

results = pd.DataFrame(rows)
results.to_csv(RESULTS_CSV, index=False)
print(f"\nSaved grid results -> {RESULTS_CSV}")
results.sort_values('Sharpe_diff_MS_AR', ascending=False).head(10)

### VISUALISE THE EDGE

In [ ]:
if len(results) == 0:
    print("No grid cells succeeded.")
else:
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    # 1. Sharpe edge per cell, sorted
    ax = axes[0, 0]
    rsorted = results.sort_values('Sharpe_diff_MS_AR')
    labels = [f"k={r.k_regimes}, w={r.winsorize_std}, c={r.coint_window}, z={r.z_window}"
              for _, r in rsorted.iterrows()]
    colors = ['#2ca02c' if v > 0 else '#d62728' for v in rsorted['Sharpe_diff_MS_AR']]
    ax.barh(range(len(rsorted)), rsorted['Sharpe_diff_MS_AR'], color=colors, alpha=0.85)
    ax.set_yticks(range(len(rsorted)))
    ax.set_yticklabels(labels, fontsize=7)
    ax.axvline(0, color='black', lw=0.8)
    ax.set_xlabel('Sharpe(MS_AR) − Sharpe(AR)')
    ax.set_title('Edge per grid cell (green = MS-AR wins)', fontweight='bold')
    ax.grid(True, axis='x', alpha=0.3)

    # 2. MS_AR vs AR sharpe scatter
    ax = axes[0, 1]
    ax.scatter(results['AR_sharpe'], results['MS_AR_sharpe'],
               c=results['Sharpe_diff_MS_AR'], cmap='RdYlGn',
               s=80, edgecolor='black')
    lim = max(abs(results[['AR_sharpe', 'MS_AR_sharpe']].values.flatten()).max(), 1)
    ax.plot([-lim, lim], [-lim, lim], 'k--', lw=1, alpha=0.5, label='y=x')
    ax.axhline(0, color='grey', lw=0.7)
    ax.axvline(0, color='grey', lw=0.7)
    ax.set_xlabel('AR Sharpe')
    ax.set_ylabel('MS_AR Sharpe')
    ax.set_title('Per-cell Sharpe — above diag = MS-AR wins', fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)

    # 3. Edge by k_regimes
    ax = axes[1, 0]
    for k in sorted(results['k_regimes'].unique()):
        sub = results[results['k_regimes'] == k]
        ax.scatter(sub['winsorize_std'] + (k - 2.5) * 0.05,
                   sub['Sharpe_diff_MS_AR'],
                   s=70, label=f'k_regimes={k}', alpha=0.8)
    ax.axhline(0, color='black', lw=0.8)
    ax.set_xlabel('winsorize_std')
    ax.set_ylabel('Sharpe edge MS_AR − AR')
    ax.set_title('Edge vs HMM hyperparams', fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)

    # 4. Trade-count comparison
    ax = axes[1, 1]
    width = 0.25
    x = np.arange(len(results))
    ax.bar(x - width, results['Baseline_n_trades'], width, label='Baseline', color='#6C757D', alpha=0.85)
    ax.bar(x,         results['AR_n_trades'],       width, label='AR',       color='#0077B6', alpha=0.85)
    ax.bar(x + width, results['MS_AR_n_trades'],    width, label='MS_AR',    color='#9D4EDD', alpha=0.85)
    ax.set_xticks(x)
    ax.set_xticklabels([f'#{i}' for i in range(len(results))], fontsize=8)
    ax.set_ylabel('Trades (per cell)')
    ax.set_title('How aggressively does each strategy trade?', fontweight='bold')
    ax.legend()
    ax.grid(True, axis='y', alpha=0.3)

    plt.suptitle(f"{PAIR_NAME} — tune grid ({TUNE_MONTHS[0]} - {TUNE_MONTHS[-1]})",
                 fontweight='bold', fontsize=13, y=1.02)
    plt.tight_layout()
    plt.show()

### PICK & PERSIST BEST CONFIG

The best cell is the one with the largest positive `Sharpe_diff_MS_AR`, with two practical guards:

- `MS_AR_sharpe > 0`  — MS-AR must actually be making money in absolute terms, not just "less negative than AR"
- `MS_AR_n_trades >= 20` — enough trades to make the result statistically meaningful in a 1-month OOS

If no cell clears both bars we keep the highest-edge cell anyway and flag it.

In [ ]:
if len(results) == 0:
    raise RuntimeError("Grid produced no successful cells — re-check data + params.")

clean = results[(results['MS_AR_sharpe'] > 0) & (results['MS_AR_n_trades'] >= 20)]
flagged = False
if len(clean) == 0:
    print("[!] No cell satisfied MS_AR_sharpe > 0 AND >= 20 trades. Falling back to highest edge.")
    clean = results.copy()
    flagged = True

best = clean.sort_values('Sharpe_diff_MS_AR', ascending=False).iloc[0]
best_cfg = {
    'pair': PAIR_NAME,
    'tune_months': TUNE_MONTHS,
    'engine': {
        'k_regimes':     int(best['k_regimes']),
        'winsorize_std': float(best['winsorize_std']),
        'coint_window':  int(best['coint_window']),
        'z_window':      int(best['z_window']),
        'train_days':    int(TRAIN_DAYS),
        'scaling':       10000,
    },
    'wfo': {
        'val_months':  int(VAL_MONTHS),
        'test_months': int(TEST_MONTHS),
        'n_trials':    int(N_TRIALS),
        'objective':   OBJECTIVE,
    },
    'metrics': {
        'MS_AR_sharpe':       float(best['MS_AR_sharpe']),
        'AR_sharpe':          float(best['AR_sharpe']),
        'Baseline_sharpe':    float(best['Baseline_sharpe']),
        'Sharpe_diff_MS_AR':  float(best['Sharpe_diff_MS_AR']),
        'PnL_diff_MS_AR_bps': float(best['PnL_diff_MS_AR_bps']),
        'MS_AR_n_trades':     int(best['MS_AR_n_trades']),
        'AR_n_trades':        int(best['AR_n_trades']),
    },
    'flagged_no_clean_cell': flagged,
}

with open(BEST_JSON, 'w') as f:
    json.dump(best_cfg, f, indent=2)
print(f"Saved best config -> {BEST_JSON}\n")
print(json.dumps(best_cfg, indent=2))

### NEXT STEPS

1. Inspect the grid table above and the heatmap. If multiple cells produce a positive `Sharpe_diff_MS_AR`, the regime model is real.
2. Copy the engine block of `audnzd_best.json` into the matching `code/notebooks/AUDNZD.ipynb` PARAMS cell.
3. Re-run the full notebook on **all 24 months** with the tuned params and confirm the edge survives out-of-sample.

If no cell shows a positive edge, the takeaway is structural: this pair / this slice doesn't have a regime structure that the gearbox can exploit. Try a different month range or treat MS-AR vs AR as essentially equivalent on this pair.